# 모델 handoff 공통 평가 리더보드

이 노트북은 `scripts/run_39_model_handoff_common_leaderboard.py`를 호출해 ZIP 복사/해제, 모델 로딩, feature schema 감사, 공통 평가 리더보드 생성을 재실행한다.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

ROOT = Path.cwd()
SCRIPT = ROOT / 'scripts' / 'run_39_model_handoff_common_leaderboard.py'
spec = importlib.util.spec_from_file_location('run39', SCRIPT)
run39 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run39)
results = run39.run_comparison(write_notebook_file=False)
results['summary']

## 공통 평가 리더보드

In [ ]:
leaderboard = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'common_pre_event_leaderboard.csv')
leaderboard

## 제외 사유

In [ ]:
schema = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'feature_schema_match_audit.csv')
schema.loc[schema['eligibility'] != 'eligible', ['owner', 'model_file', 'task_family', 'eligibility', 'reason', 'missing_feature_count']].head(50)

## Front Gate 리더보드

In [ ]:
front_leaderboard = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'front_gate_leaderboard.csv')
front_leaderboard

## Front Gate 제외 사유

In [ ]:
front_schema = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'front_gate_schema_match_audit.csv')
front_schema.loc[~front_schema['eligibility'].str.startswith('eligible'), ['owner', 'model_file', 'task_family', 'eligibility', 'reason']].head(50)

## 검증

In [ ]:
summary = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'evaluation_summary.csv').iloc[0]
predictions = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'common_pre_event_predictions.csv')
expected = int(summary['evaluation_rows']) * int(summary['leaderboard_model_count'])
assert len(predictions) == expected, (len(predictions), expected)
front_summary = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'front_gate_evaluation_summary.csv').iloc[0]
front_predictions = pd.read_csv(ROOT / '10_모델비교' / '04_산출물' / 'front_gate_predictions.csv')
front_expected = int(front_summary['evaluation_rows']) * int(front_summary['leaderboard_model_count'])
assert len(front_predictions) == front_expected, (len(front_predictions), front_expected)
print({'pre_event_prediction_rows': len(predictions), 'pre_event_expected_rows': expected, 'front_gate_prediction_rows': len(front_predictions), 'front_gate_expected_rows': front_expected})